In [6]:
#SEnet
import torch
import torch.nn as nn
# from torchsummary import summary
import numpy as np
import sys
import torch.nn.functional as F

if torch.cuda.is_available():
   device = torch.device("cuda")
else:
   device = torch.device("cpu")

class SE_Module(nn.Module):

    def __init__(self, in_channels, ratio=4, dim=1):
        super(SE_Module, self).__init__()
        self.dim = dim
        if self.dim == 1:
            self.squeeze = nn.AdaptiveAvgPool1d(1)
        else:
            self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(in_features=in_channels, out_features=in_channels // ratio),
            nn.ReLU(inplace=True),
            nn.Linear(in_features=in_channels // ratio, out_features=in_channels),
            nn.Sigmoid()
        )


    def forward(self, x):
        identity = x

        out = self.squeeze(x)
        out = out.reshape(out.shape[0], out.shape[1])
        scale = self.excitation(out)
        if self.dim == 1:
            scale = scale.reshape(scale.shape[0], scale.shape[1], 1)
        else:
            scale = scale.reshape(scale.shape[0], scale.shape[1], 1, 1)

        return identity * scale.expand_as(identity)

class ResBlock2d(nn.Module):
    """docstring for ResBlock2d"""
    def __init__(self, filters):
        super(ResBlock2d, self).__init__()

        self.conv2d0 = nn.Sequential(
            nn.Conv2d(1, filters // 2, kernel_size = (20, 1), padding = ((20 - 1) // 2, 0) ),
            nn.BatchNorm2d(filters // 2),
            nn.LeakyReLU(0.3)
        )
       
        self.conv2d1 = nn.Sequential(
            nn.Conv2d(filters // 2, filters, kernel_size = (17, 1), padding = ((17 - 1) // 2, 0) ),
            nn.BatchNorm2d(filters),
            nn.LeakyReLU(0.3)
        )

        self.conv2d2 = nn.Sequential(
            nn.Conv2d(filters, filters, kernel_size = (11, 1), padding = ((11 - 1) // 2, 0) ),
            nn.BatchNorm2d(filters),
            nn.LeakyReLU(0.3)
        )

        self.conv2d3 = nn.Sequential(
            nn.Conv2d(filters, filters, kernel_size = (5, 1), padding = ((5 - 1) // 2, 0) ),
            nn.BatchNorm2d(filters),
            nn.LeakyReLU(0.3)
        )

        self.SE = SE_Module(filters, dim=2)

        self.conv2d4 = nn.Sequential(
            nn.Conv2d(filters // 2, filters, kernel_size = (1, 1), padding = (0, 0) ),
            nn.BatchNorm2d(filters),
            nn.LeakyReLU(0.3)
        )

        self.leakyrule = nn.LeakyReLU(0.3)

    def forward(self, x):
        x = self.conv2d0(x)
        output = x
        output = self.conv2d1(output)
        output = self.conv2d2(output)
        output = self.conv2d3(output)
        output = self.SE(output)

        x = self.conv2d4(x)

        output = output + x

        output = self.leakyrule(output)

        return output

class ResBlock1d(nn.Module):
    """docstring for ResBlock2d"""
    def __init__(self, filters):
        super(ResBlock1d, self).__init__()
        filters0 = filters
        if filters != 4:
            filters0 = filters // 2

        self.conv1d1 = nn.Sequential(
            nn.Conv1d(filters0, filters, kernel_size = 17, padding = (17 - 1) // 2),
            nn.BatchNorm1d(filters),
            nn.LeakyReLU(0.3)
        )

        self.conv1d2 = nn.Sequential(
            nn.Conv1d(filters, filters, kernel_size = 11, padding = (11 - 1) // 2),
            nn.BatchNorm1d(filters),
            nn.LeakyReLU(0.3)
        )

        self.conv1d3 = nn.Sequential(
            nn.Conv1d(filters, filters, kernel_size = 5, padding = (5 - 1) // 2),
            nn.BatchNorm1d(filters),
            nn.LeakyReLU(0.3)
        )

        self.SE = SE_Module(filters, dim=1)

        self.conv1d4 = nn.Sequential(
            nn.Conv1d(filters0, filters, kernel_size = 1, padding = 0),
            nn.BatchNorm1d(filters),
            nn.LeakyReLU(0.3)
        )

        self.leakyrule = nn.LeakyReLU(0.3)
        self.avp = nn.AvgPool1d(3)

    def forward(self, x):
        output = x
        output = self.conv1d1(output)
        output = self.conv1d2(output)
        output = self.conv1d3(output)
        output = self.SE(output)

        x = self.conv1d4(x)

        output = output + x

        output = self.leakyrule(output)
        output = self.avp(output)

        return output
     
class SEnet(nn.Module):
    """docstring for SEnet"""
    def __init__(self):
        super(SEnet, self).__init__()
        self.blk1 = ResBlock2d(4)
        self.blk2 = ResBlock1d(4)
        self.blk3 = ResBlock1d(8)
        self.blk4 = ResBlock1d(16)
        self.gap = nn.AdaptiveMaxPool1d(1)


    def forward(self, x):
        # output = torch.sin(x)
        # x += output
        x = self.blk1(x)
        x = torch.squeeze(x, dim = 3)
        x = self.blk2(x)
        x = self.blk3(x)
        x = self.blk4(x)
        x = self.gap(x)
        return x

class mSEnet(nn.Module):
    """docstring for mSEnet"""
    def __init__(self):
        super(mSEnet, self).__init__()
        self.senet = SEnet()

    def forward(self, x):
        x = torch.unsqueeze(x, dim = 1) 
        x = torch.unsqueeze(x, dim = -1)
        x = self.senet(x)
        
        return x

In [7]:
#basemodel_cab

def xavier_init(m):
    if type(m) == nn.Linear:
        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            m.bias.data.fill_(0.0)


class LinearLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.clf = nn.Sequential(nn.Linear(in_dim, out_dim))
        self.clf.apply(xavier_init)

    def forward(self, x):
        x = self.clf(x)
        return x
    
class PDF3(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_class, dropout):  # in_dim是有12个5000的列表，hidden_dim是[16],num_class是7
        super().__init__()
        self.views = len(in_dim)  # 12��12������
        self.classes = num_class  # ��������Ĭ��Ϊ7
        self.dropout = dropout

        self.FeatureEncoder = nn.ModuleList(
            [mSEnet() for view in range(self.views)])  # ������������������views��mSE���磨ÿ������ƥ��һ��������

        self.MMClasifier = []
        self.MMClasifier.append(LinearLayer(self.views * hidden_dim[-1], 2))  # ��������1
        self.MMClasifier = nn.Sequential(*self.MMClasifier)

        self.MMClasifier5 = []
        self.MMClasifier5.append(LinearLayer(self.views * hidden_dim[-1], 5))  # ��������2
        self.MMClasifier5 = nn.Sequential(*self.MMClasifier5)

        self.MMClasifier7 = []
        self.MMClasifier7.append(LinearLayer(self.views * hidden_dim[-1], 7))  # ��������3
        self.MMClasifier7 = nn.Sequential(*self.MMClasifier7)

    # def forward(self, data_list, label2=None, label5=None, label7=None, dataset='PTB'):
    def forward(self, data_list, label2=None, label5=None, label7=None, dataset='PTBXL'):
        # �����
        FeatureInfo, feature, feature_lead, TCPLogit, TCPConfidence, lead_weight, TCPpreLogit = dict(), dict(), dict(), dict(), dict(), dict(), dict()
        view_weight_sigmoid = torch.empty(3, 3)

        for view in range(self.views):
            feature[view] = self.FeatureEncoder[view](data_list[view])
            #将data_list[view]融合为单导联
            feature[view] = feature[view].flatten(start_dim=1, end_dim=2)
        # ���ڿ�����ط�=
        MIfeature, MIfeature_1 = dict(), dict()

        MMfeature = torch.cat([i for i in feature.values()], dim=1)
        MMfeature = F.dropout(MMfeature, self.dropout, training=self.training)

        MMlogit = self.MMClasifier(MMfeature)
        MMlogit5 = self.MMClasifier5(MMfeature)
        MMlogit7 = self.MMClasifier7(MMfeature)

        criterion0 = torch.nn.CrossEntropyLoss(reduction='none')
        criterion = torch.nn.BCEWithLogitsLoss()  # BCEloss���ڶ����ཻ���أ��ú���������BCE Loss�Լ�Sigmoid�����㣬
        if 'PTBXL' in dataset:
            Loss2 = torch.mean(criterion0(MMlogit, label2))
            Loss5 = torch.mean(criterion(MMlogit5, label5.to(torch.float)))
            Loss7 = torch.mean(criterion(MMlogit7, label7.to(torch.float)))
        else:
            Loss2 = torch.mean(criterion0(MMlogit, label2))
            Loss5 = torch.mean(criterion0(MMlogit5, label5))
            Loss7 = torch.mean(criterion0(MMlogit7, label7))

        # MMLoss = 0.013 * Loss2 + 0.19 * Loss5 + 0.79 * Loss7
        MMLoss = 0.1 * Loss2  + 0.9 * Loss7
        return MMLoss, MMlogit7


In [8]:
import numpy as np, os
from sklearn.metrics import *
# from Metric import *
import sys

def MacroAUC(output, label):
    y_pred = output
    y_true = label
    num_instance,num_class = y_pred.shape
    count = np.zeros((num_class,1))
    num_P_instance =  np.zeros((num_class,1)) 
    num_N_instance =  np.zeros((num_class,1)) 
    auc = np.zeros((num_class,1))
    count_valid_label = 0
    for  i in range(num_class):
        num_P_instance[i,0] = sum(y_true[:,i] == 1)
        num_N_instance[i,0] = num_instance - num_P_instance[i,0]
        if num_P_instance[i,0] == 0 or num_N_instance[i,0] == 0:
            auc[i,0] = 0
            count_valid_label = count_valid_label + 1
        else:
            temp_P_Outputs = np.zeros((int(num_P_instance[i,0]), num_class))
            temp_N_Outputs = np.zeros((int(num_N_instance[i,0]), num_class))
  
            temp_P_Outputs[:,i] = y_pred[y_true[:,i]==1,i]
            temp_N_Outputs[:,i] = y_pred[y_true[:,i]==0,i]    
            for m in range(int(num_P_instance[i,0])):
                for n in range(int(num_N_instance[i,0])):
                    if(temp_P_Outputs[m,i] > temp_N_Outputs[n,i] ):
                        count[i,0] = count[i,0] + 1
                    elif(temp_P_Outputs[m,i] == temp_N_Outputs[n,i]):
                        count[i,0] = count[i,0] + 0.5
            
            auc[i,0] = count[i,0]/(num_P_instance[i,0]*num_N_instance[i,0])  
    macroAUC1 = sum(auc)/(num_class-count_valid_label)
    return  float(macroAUC1)

def ptbxl_Result(class_num, y_pred, y_test, baseline = 0.5):
    output_labels=[]
    
    for i,key in enumerate(y_pred):
        output_label=[]
        for j in range(len(key)):
            if(key[j] >= baseline):
                output_label.append(1)
            else:
                output_label.append(0)
        output_label=np.array(output_label)
        output_labels.append(output_label)
    
    output_labels=np.array(output_labels)
    output=output_labels
    label=y_test

    auc = MacroAUC(output,label) 
 
    y_pred=np.where(output>0.5,1,0)
    acc=accuracy_score(y_pred,label)

    return acc, auc

def Sen(con_mat,n=4):
    
    sen = []
    for i in range(n):
        tp = con_mat[i][i]
        fn = np.sum(con_mat[i,:]) - tp
        sen1 = tp / (tp + fn)
        sen.append(sen1)
        
    return sen

def Spe(con_mat,n=4):
    
    spe = []
    temp = 0
    for i in range(n):
        temp += con_mat[i][i]
    for i in range(n):
        number = np.sum(con_mat[:,:])
        tp = con_mat[i][i]
        fn = np.sum(con_mat[i,:]) - tp
        fp = np.sum(con_mat[:,i]) - tp
        tn = number - tp - fn - fp
        spe1 = (temp - tp) / (tn + fp )
        #print(spe1)
        spe.append(spe1)
    
    return spe
    
def ACC(con_mat,n=4):
    
    acc = []
    number = np.sum(con_mat[:,:])
    temp = 0
    for i in range(n):
        temp += con_mat[i][i]
    acc = temp / number    
    return acc

def ptb_Result(class_num,y_pred,y_test):
    y_pred = np.argmax(y_pred,axis=1)
    if(int(class_num)==7):
        target_names = ['AMI','ASMI','ALMI','IMI','ILMI','other','NORM']
    else:
        print("num_class error!")
        sys.exit()

    con_mat = confusion_matrix(y_test, y_pred)  # 把输入转变为混淆矩阵
    n = con_mat.shape[0]
    sen = Sen(con_mat,n)
    spe = Spe(con_mat,n)
    acc = ACC(con_mat,n)
    return np.sum(sen)/n, np.sum(spe)/n, acc

In [10]:
#dataset
from torch.utils.data import Dataset
import torch
import numpy as np

cuda = True if torch.cuda.is_available() else False

class matDataset(Dataset):#X是输入数据集，Y、Y5、Y7分别是三种分类的标签
    def __init__(self, X, Y, Y5, Y7):
        self.X = X
        self.Y = Y
        self.Y5 = Y5
        self.Y7 = Y7

    def __getitem__(self, index):#应该是拿出指定导联的数据？
        idx = index % len(self.Y)
        data = []
        for x in self.X:
            data.append(x[idx])
        x = data
        y = self.Y[idx]
        y5 = self.Y5[idx]
        y7 = self.Y7[idx]
        
        if cuda:
            for i in range(len(x)):
                x[i] = x[i].cuda()
            y = y.cuda()
            y5 = y5.cuda()
            y7 = y7.cuda()
        return x, y, y5, y7

    def __len__(self):
        return len(self.Y)

In [11]:
#pytorchtools
import numpy as np
import torch
#提前结束训练的代码文件
class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pt', trace_func=print):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
                            Default: 7
            verbose (bool): If True, prints a message for each validation loss improvement. 
                            Default: False
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                            Default: 0
            path (str): Path for the checkpoint to be saved to.
                            Default: 'checkpoint.pt'
            trace_func (function): trace print function.
                            Default: print            
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.acc_min = 0
        self.delta = delta
        self.path = path
        self.trace_func = trace_func
    def __call__(self, val_loss, model,acc_score):

        score = -val_loss
        # score = -acc_score#准确率作为筛选

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model,acc_score)
        elif score < self.best_score + self.delta:
            self.counter += 1
            self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        # elif score > self.best_score + self.delta:
        #     self.counter += 1
        #     self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
        #     if self.counter >= self.patience:
        #         self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model ,acc_score)
            self.counter = 0

    def save_checkpoint(self, val_loss, model,acc_score):
        if self.verbose:
            self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).')
        self.val_loss_min = val_loss
        #     self.trace_func(f'Validation loss decreased ({self.acc_min:.6f} --> {acc_score:.6f}).')
        # self.acc_min = acc_score

In [12]:

import pandas as pd
import numpy as np
import wfdb
import random as rn
import ast
import os
import glob
import sys
#读取ptbxl数据集的文件，这里的df是取了需要折数的原database的文件
def load_ptbxl_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]#wfdb.rdsamp是专门读取ECG信号的函数
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data
#为norm和MI的二分类问题设置独热编码
def one_hot_2(y_test):
    test = []
    for i in range(len(y_test.values)):
        if 'NORM' in y_test.values[i]:
            test.append(0)
        elif 'MI' in y_test.values[i]:
            test.append(1)
    return np.array(test)
#为五分类问题设置独热编码
def one_hot_5(y_test):
    labels = np.zeros((len(y_test), 5))
    for i in range(len(y_test.values)):
        if len(y_test.values[i])==0:
            continue
        if 'NORM' in y_test.values[i]:
            labels[i,4]=1
        if 'AMI' in y_test.values[i]:
            labels[i,0]=1
        if 'ASMI' in y_test.values[i]:
            labels[i,1]=1
        if 'ALMI' in y_test.values[i]:
            labels[i,2]=1
        if 'other'in y_test.values[i]:
            labels[i,3]=1
    return labels
##为七分类问题设置独热编码
def one_hot_7(y_test):
    labels = np.zeros((len(y_test), 7))
    for i in range(len(y_test.values)):
        if len(y_test.values[i])==0:
            continue
        if 'NORM' in y_test.values[i]:
            labels[i,6]=1
        if 'AMI' in y_test.values[i]:
            labels[i,0]=1
        if 'ASMI' in y_test.values[i]:
            labels[i,1]=1
        if 'ALMI' in y_test.values[i]:
            labels[i,2]=1
        if 'IMI' in y_test.values[i]:
            labels[i,3]=1
        if 'ILMI' in y_test.values[i]:
            labels[i,4]=1
        if 'other' in y_test.values[i]:
            labels[i,5]=1
    return labels


In [13]:
def read_ptbxl(num_class,fold_num):#功能就是读取数据和标签，进行标签独热编码
    path = '../'
    sampling_rate=500
# SCP-ECG文件是一种用于存储心电图数据的文件格式 #new_ptbxl_database.csv存储了标签
    Y = pd.read_csv(path+'new_ptbxl_database.csv', index_col='ecg_id',encoding = 'gb2312')
    Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))#取出scp_codes字典中的变量并赋值
    agg_df = pd.read_csv(path+'scp_statements_257x.csv', index_col=0)#scp_statements_257x.csv'包含了分类任务里面包含哪几类的信息
    agg_df = agg_df[agg_df.diagnostic == 1]#选出用于分类中的标签

    def aggregate_diagnostic(y_dic):#选出当前用于分类的的标签
        tmp = []
        for key in y_dic.keys():
            if key in agg_df.index:
                if int(num_class) == 2:
                    tmp.append(agg_df.loc[key].diagnostic_class)
                elif int(num_class) == 5:
                    tmp.append(agg_df.loc[key].diagnostic_5class)
                elif int(num_class) == 7:
                    tmp.append(agg_df.loc[key].diagnostic_7class)
        return list(set(tmp))

    Y['diagnostic_class'] = Y.scp_codes.apply(aggregate_diagnostic)#筛选标签
    # Y是读取的文件数据，Y.strat_fold是第几折的数据
    strat_fold_train=Y[Y.strat_fold != fold_num]
    X_train = load_ptbxl_data(strat_fold_train, sampling_rate, path)#心电图信号数据
    y_train = strat_fold_train.diagnostic_class

    strat_fold_test=Y[Y.strat_fold == fold_num]
    X_test = load_ptbxl_data(strat_fold_test, sampling_rate, path)
    y_test = strat_fold_test.diagnostic_class
    #独热编码设置，出来是一个（class*输入向量长度）大小的矩阵，里面是独热编码
    if(int(num_class)==2):
        y_train = one_hot_2(y_train)
        y_test = one_hot_2(y_test)
    elif(int(num_class)==5):
        y_train = one_hot_5(y_train)
        y_test = one_hot_5(y_test)
    elif(int(num_class)==7):
        y_train = one_hot_7(y_train)
        y_test = one_hot_7(y_test)

    X_train = np.array(X_train)#输入数据，即输入的波形图
    X_test = np.array(X_test)
    
    def re_format(X):#将输入的数据的每个数据转换为数组格式，存放在列表里
        X = X.transpose(2,0,1).tolist()
        X_te = []
        for i,a in enumerate(X):#enumerate用于迭代，对矩阵的每一行操作
            c = []
            for j,b in enumerate(a):#这里是对每一行的每个元素操作
                c.append(np.array(b))
            X_te.append(np.array(c))
        return X_te
        
    return re_format(X_train),y_train,re_format(X_test),y_test

In [18]:
import os, sys
import numpy as np
import torch
import torch.nn.functional as F
# from basemodel_uni import UNI
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cuda = True if torch.cuda.is_available() else False#使用GPU
# num_view是导联数，data_folder是数据集名称，num_class是分类数，num_fold是第几个文件夹
def prepare_trte_data(data_folder, num_class, num_fold, num_view = 12):
    if 'PTBXL' in data_folder:
        data_tr_list, labels_tr, data_te_list, labels_te = read_ptbxl(num_class, num_fold)
# read_ptbxl可以读取数据的训练集和测试集的数据列表和标签列表
    elif 'PTB' in data_folder:
        data_tr_list, labels_tr, data_te_list, labels_te = read_ptb(num_class, num_fold)

    num_tr = data_tr_list[0].shape[0]  # 训练数据集的第一维个数
    num_te = data_te_list[0].shape[0]
# 下面一段是为了转tensor
    data_mat_list = []
    for i in range(num_view):
        data_mat_list.append(np.concatenate((data_tr_list[i], data_te_list[i]), axis=0))  # 在每个导联下，数组按照第一维合并训练数据集和测试数据集

    data_tensor_list = []
    for i in range(len(data_mat_list)):  # 把十二导联的总数据集合并集变成tensor
        data_tensor_list.append(torch.FloatTensor(data_mat_list[i]))

    idx_dict = {}
    idx_dict["tr"] = list(range(num_tr))  # 训练数据集的第一维长度列表
    idx_dict["te"] = list(range(num_tr, (num_tr+num_te)))  # 终止是总数据集的第一维长度列表
    data_train_list = []  # 存放训练数据集
    data_test_list = []  # 存放测试数据集

    for i in range(len(data_tensor_list)):
        data_train_list.append(data_tensor_list[i][idx_dict["tr"]])
        data_test_list.append(data_tensor_list[i][idx_dict["te"]])

    labels = np.concatenate((labels_tr, labels_te))  # 合并总标签集
    return data_train_list, data_test_list, idx_dict, labels  # tensor格式的训练和测试数据集，训练集和测试集个数索引（用于分开标签，总标签）
# 建模后运行
def run(model, testonly, num_class = 7, fold_num = 10, dataset = 'PTBXL'):
    num_epoch = 200
    lr = 1e-4
    # 数据读取

    data_tr_list, data_test_list, trte_idx, labels_trte = prepare_trte_data(dataset, 2, fold_num, num_view = 12)
    data_tr_list5, data_test_list5, trte_idx5, labels_trte5 = prepare_trte_data(dataset, 5, fold_num, num_view = 12)
    data_tr_list7, data_test_list7, trte_idx7, labels_trte7 = prepare_trte_data(dataset, 7, fold_num, num_view = 12)
# 标签转tensor
    labels_tr_tensor = torch.LongTensor(labels_trte[trte_idx["tr"]])
    labels_te_tensor = torch.LongTensor(labels_trte[trte_idx["te"]])
    labels_tr_tensor5 = torch.LongTensor(labels_trte5[trte_idx["tr"]])
    labels_te_tensor5 = torch.LongTensor(labels_trte5[trte_idx["te"]])
    labels_tr_tensor7 = torch.LongTensor(labels_trte7[trte_idx["tr"]])
    labels_te_tensor7 = torch.LongTensor(labels_trte7[trte_idx["te"]])
    # 转化为dataloader能识别的dataset
    dataset_tr = matDataset(data_tr_list, labels_tr_tensor, labels_tr_tensor5, labels_tr_tensor7)
    dataset_te = matDataset(data_test_list, labels_te_tensor, labels_te_tensor5, labels_te_tensor7)

    train_loader = torch.utils.data.DataLoader(dataset_tr, batch_size=128)
    test_loader = torch.utils.data.DataLoader(dataset_te, batch_size=512)
    # train_loader = torch.utils.data.DataLoader(dataset_tr, batch_size=24)  # 在这里添加shuffle
    # test_loader = torch.utils.data.DataLoader(dataset_te, batch_size=128)

    # train_loader = torch.utils.data.DataLoader(dataset_tr, batch_size=128,shuffle = True)#在这里添加shuffle
    # test_loader = torch.utils.data.DataLoader(dataset_te, batch_size=512)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    # optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)

    # 复现当前论文的结果
    if testonly:
        model = torch.load('./{}/Best_Model_KFold{}.pt'.format(dataset, fold_num))
        te_prob = []
        for batch_idx, (batch, y, y5, y7) in enumerate(test_loader):
            with torch.no_grad():
                elogit = model(batch, y, y5, y7)[1]
                prob = torch.sigmoid(elogit).data.cpu().numpy()

                if len(te_prob) == 0:
                    te_prob = prob
                else:
                    te_prob = np.concatenate((te_prob, prob),axis=0)

        if 'PTBXL' in dataset:
            eval_acc, eval_auroc = ptbxl_Result(num_class, te_prob, labels_trte7[trte_idx["te"]])
            print('Eval Acc: {:.6f}, Eval Auroc: {:.6f}'.format(eval_acc, eval_auroc))
        elif 'PTB' in dataset:
            eval_sen, eval_spe, eval_acc = ptb_Result(num_class, te_prob, labels_trte7[trte_idx["te"]])
            print('Eval Sen: {:.6f}, Eval Spe: {:.6f}, Eval Acc: {:.6f}'.format(eval_sen, eval_spe, eval_acc))

    # 训练+验证    
    else:
        print("\nTraining...")

        train_losses, train_acces, train_aurocs = [], [], []
        eval_losses, eval_acces, eval_aurocs = [], [], []

        early_stopping = EarlyStopping(patience = 15, verbose = True)
        # print(2)
        for epoch in range(num_epoch+1):
            # train the model #
            train_loss = 0
            tr_prob = []

            model.train()
            cnt = 0
            for batch_idx, (batch, y, y5, y7) in enumerate(train_loader):
                cnt += 1
                optimizer.zero_grad()
                loss, logit = model(batch, y, y5, y7, dataset)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                prob = torch.sigmoid(logit).data.cpu().numpy()

                if len(tr_prob) == 0:
                    tr_prob = prob
                else:
                    tr_prob = np.concatenate((tr_prob, prob),axis=0)

            if 'PTBXL' in dataset:
                train_acc, train_auroc = ptbxl_Result(num_class, tr_prob, labels_trte7[trte_idx["tr"]])
            elif 'PTB' in dataset:
                train_sen, train_spe, train_acc = ptb_Result(num_class, tr_prob, labels_trte7[trte_idx["tr"]])

            # eval the model #
            eval_loss = 0
            te_prob = []

            model.eval()
            ecnt = 0
            for batch_idx, (batch, y, y5, y7) in enumerate(test_loader):
                ecnt += 1
                with torch.no_grad():
                    eloss, elogit = model(batch, y, y5, y7, dataset)

                    eval_loss += eloss.item()
                    prob = torch.sigmoid(elogit).data.cpu().numpy()

                    if len(te_prob) == 0:
                        te_prob = prob
                    else:
                        te_prob = np.concatenate((te_prob, prob),axis=0)

            if 'PTBXL' in dataset:
                eval_acc, eval_auroc = ptbxl_Result(num_class, te_prob, labels_trte7[trte_idx["te"]])
                print('epoch: {}, Train Loss: {:.6f}, Train Acc: {:.6f}, Train Auroc: {:.6f}, Eval Loss: {:.6f}, Eval Acc: {:.6f}, Eval Auroc: {:.6f}'
                .format(epoch, train_loss / cnt, train_acc, train_auroc, eval_loss / ecnt, eval_acc, eval_auroc))
            elif 'PTB' in dataset:
                eval_sen, eval_spe, eval_acc = ptb_Result(num_class, te_prob, labels_trte7[trte_idx["te"]])
                print('epoch: {}, Train Loss: {:.6f}, Train Sen : {:.6f}, Train Spe: {:.6f}, Train Acc: {:.6f}, Eval Loss: {:.6f}, Eval Sen: {:.6f}, Eval Spe: {:.6f}, Eval Acc: {:.6f}'
                .format(epoch, train_loss / cnt, train_sen, train_spe, train_acc, eval_loss / ecnt, eval_sen, eval_spe, eval_acc))

            early_stopping(eval_loss / ecnt, model, eval_acc)
            confusion_ptbxl = multilabel_confusion_matrix(labels_trte7[trte_idx["te"]], te_prob)
            for i, cm in enumerate(confusion_ptbxl):
                print(f'Confusion matrix for label {i}:')
                print(cm)
            if early_stopping.early_stop:
                print(te_prob)
                # confusion_ptbxl = confusion_matrix(te_prob,labels_trte7[trte_idx["te"]])


                break

def train(testonly, num_class = 7, dataset = 'PTBXL'):
    if 'PTBXL' in dataset:
        folds = 10
        dataset_list = [5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000]
    elif 'PTB' in dataset:
        folds = 5  # 交叉验证折数
        dataset_list = [600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600]  # 每个导联是600长度
    else:
        print("dataset error!")
        sys.exit()
    # 交叉验证
    for i in range(folds):
        print('KFold: {}'.format(i + 1))

        # # 因为MCA-net的第四折保存的模型损坏了所以没办法读取，这里直接输出当时的结果。
        # if 'PTBXL' not in dataset and int(i + 1) == 4 and testonly == True:
        #     print('Eval Sen: 0.293538, Eval Spe: 0.347262, Eval Acc: 0.348828')
        #     continue

        dim_list = dataset_list
        hidden_dim = [16]
        model = PDF3(dim_list, hidden_dim, num_class, dropout=0.5)  # 建模型
        # model = UNI(dim_list, hidden_dim, num_class, dropout=0.5)
        # model.cuda()  # 放入模型到GPU
        run(model, testonly, num_class, int(i + 1), dataset)  # 运行模型


In [21]:
import sys

if __name__ == '__main__':
    # Parse arguments.
    test = False

    print('Running training code...')
    train(test, 7, 'PTBXL')
    print('Done.')

Running training code...
KFold: 1


FileNotFoundError: [Errno 2] No such file or directory: '../new_ptbxl_database.csv'